# Análisis histórico de acciones e ingresos: Tesla y GameStop con Python

**Entrega:** Proyecto final de IBM — análisis histórico de acciones/ingresos y creación de un panel de control.

En este Notebook se completan los **seis ejercicios** solicitados por la rúbrica. Las evidencias de cada pregunta se guardan automáticamente como archivos PNG dentro de la carpeta `evidencias_IBM/`, para adjuntarlas por separado en la evaluación.  

**Fuentes:** cotizaciones de Yahoo Finance mediante `yfinance` (descarga actual al ejecutarlo); ingresos trimestrales de las **copias históricas HTML de IBM**. El alcance de los gráficos se limita a mediados de 2021 para comparar con los gráficos del ejercicio original.

**Importante:** ejecuta el Notebook con conexión a Internet; las imágenes de evidencia se crean con los datos realmente obtenidos durante esa ejecución. La última celda empaqueta las seis imágenes en un ZIP.

## Preparación del entorno

Si estás usando Google Colab, ejecuta esta celda y luego todas las siguientes en orden.

In [ ]:
%pip -q install yfinance beautifulsoup4 pandas requests matplotlib nbformat

In [ ]:
import re
import time
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED

import yfinance as yf
import pandas as pd
import requests
from bs4 import BeautifulSoup
import matplotlib.pyplot as plt
from matplotlib import gridspec
from IPython.display import display

CARPETA = Path("evidencias_IBM")
CARPETA.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 20)
print("Entorno listo. Evidencias en:", CARPETA.resolve())

## Funciones de extracción, visualización y evidencias

Esta sección prepara las funciones de apoyo para que las siguientes seis preguntas se centren en el código y los resultados requeridos. La limpieza elimina los símbolos `$` y las comas de la columna `Revenue` y descarta valores vacíos.

In [ ]:
def obtener_html(url):
    """Descarga HTML para BeautifulSoup, con hasta tres intentos."""
    encabezados = {"User-Agent": "Mozilla/5.0 (compatible; IBM-educational-project/1.0)"}
    ultimo_error = None
    for intento in range(3):
        try:
            respuesta = requests.get(url, headers=encabezados, timeout=40)
            respuesta.raise_for_status()
            if not respuesta.text.strip():
                raise ValueError("La página devolvió HTML vacío")
            return respuesta.text
        except (requests.RequestException, ValueError) as error:
            ultimo_error = error
            if intento < 2:
                time.sleep(2 * (intento + 1))
    raise RuntimeError(f"No se pudo descargar el HTML: {url}. Error: {ultimo_error}")


def extraer_ingresos_trimestrales(html_data, nombre_empresa):
    """Web scraping con BeautifulSoup de la tabla trimestral, con orden original."""
    soup = BeautifulSoup(html_data, "html.parser")
    filas = []
    for tabla in soup.find_all("table"):
        encabezados = " ".join(c.get_text(" ", strip=True) for c in tabla.find_all("th"))
        if f"{nombre_empresa.lower()} quarterly revenue" not in encabezados.lower():
            continue
        for fila in tabla.find_all("tr"):
            columnas = fila.find_all("td")
            if len(columnas) < 2:
                continue
            fecha = columnas[0].get_text(" ", strip=True)
            ingresos = columnas[1].get_text(" ", strip=True)
            if re.fullmatch(r"\d{4}-\d{2}-\d{2}", fecha):
                filas.append({"Date": fecha, "Revenue": ingresos})
        break
    if not filas:
        raise RuntimeError(f"No se encontró la tabla 'Quarterly Revenue' de {nombre_empresa}.")
    df = pd.DataFrame(filas, columns=["Date", "Revenue"])
    df["Revenue"] = (df["Revenue"].astype(str)
                    .str.replace(r"[$,]", "", regex=True)
                    .str.strip())
    df = df[df["Revenue"].ne("")].copy()
    df["Revenue"] = pd.to_numeric(df["Revenue"], errors="coerce")
    df.dropna(subset=["Revenue"], inplace=True)
    df["Revenue"] = df["Revenue"].astype(int)
    df.reset_index(drop=True, inplace=True)
    return df


def guardar_evidencia_tabla(numero, titulo, codigo, dataframe, archivo):
    """Construye evidencia PNG de código utilizado + cinco filas resultantes."""
    df = dataframe.copy()
    for columna in df.columns:
        if pd.api.types.is_datetime64_any_dtype(df[columna]):
            df[columna] = pd.to_datetime(df[columna]).dt.strftime("%Y-%m-%d")
        elif pd.api.types.is_float_dtype(df[columna]):
            df[columna] = df[columna].map(lambda x: f"{x:.6f}" if pd.notna(x) else "")
        else:
            df[columna] = df[columna].astype(str)
    fig = plt.figure(figsize=(17, 6.0), facecolor="white")
    gs = gridspec.GridSpec(3, 1, height_ratios=[0.55, 1.1, 2.7], hspace=0.06)
    ax_t = fig.add_subplot(gs[0]); ax_t.axis("off")
    ax_t.text(0.02, 0.65, f"PREGUNTA {numero}  |  {titulo}", fontsize=18,
              fontweight="bold", color="#142b43", va="center")
    ax_c = fig.add_subplot(gs[1]); ax_c.set_facecolor("#f3f6f9")
    ax_c.set_xlim(0, 1); ax_c.set_ylim(0, 1)
    for s in ax_c.spines.values(): s.set_visible(False)
    ax_c.set_xticks([]); ax_c.set_yticks([])
    ax_c.text(0.018, 0.5, codigo, family="monospace", fontsize=12, color="#182c41", va="center")
    ax_tab = fig.add_subplot(gs[2]); ax_tab.axis("off")
    ax_tab.text(0.02, 0.99, "Resultado obtenido al ejecutar el Notebook:",
                color="#3f5269", fontsize=12, va="top", transform=ax_tab.transAxes)
    columnas = ["Índice"] + list(df.columns)
    valores = [[str(ix)]+[str(x) for x in row] for ix, row in df.iterrows()]
    if len(columnas) <= 3:
        widths = [0.11] + [0.35] * (len(columnas) - 1)
    else:
        widths = [0.06] + [0.94 / (len(columnas)-1)] * (len(columnas)-1)
    tab = ax_tab.table(cellText=valores, colLabels=columnas, loc="center",
                       cellLoc="center", colLoc="center", colWidths=widths,
                       bbox=[0.01, 0.02, 0.98, 0.84])
    tab.auto_set_font_size(False)
    tab.set_fontsize(9.3 if len(columnas)>5 else 11)
    for (r,c), celda in tab.get_celld().items():
        celda.set_edgecolor("#dee5ec"); celda.set_linewidth(0.6)
        if r == 0:
            celda.set_facecolor("#dbeafe")
            celda.get_text().set_weight("bold")
            celda.get_text().set_color("#163458")
        elif r%2 == 0:
            celda.set_facecolor("#f5f8fb")
    ruta = CARPETA / archivo
    fig.savefig(ruta, dpi=160, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print("Evidencia guardada:", ruta)


def make_graph(stock_data, revenue_data, stock):
    """Gráficos históricos de la acción y los ingresos, como solicita IBM."""
    if stock_data.empty or revenue_data.empty:
        raise ValueError(f"No hay datos suficientes para graficar {stock}.")
    precios = stock_data.copy()
    ingresos = revenue_data.copy()
    precios["Date"] = pd.to_datetime(precios["Date"], utc=True).dt.tz_localize(None)
    ingresos["Date"] = pd.to_datetime(ingresos["Date"])
    # El proyecto original visualiza los datos anteriores al 14 de junio de 2021.
    limite = pd.Timestamp("2021-06-14")
    precios = precios[precios["Date"] <= limite].copy()
    ingresos = ingresos[ingresos["Date"] <= limite].copy()
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 8.2), sharex=True,
                                    gridspec_kw={"hspace": 0.31})
    fig.set_facecolor("white")
    fig.suptitle(f"{stock} - Acciones e ingresos históricos", fontsize=18,
                 fontweight="bold", y=0.985)
    ax1.plot(precios["Date"], precios["Close"], color="#2563eb", linewidth=1.5)
    ax1.set_title(f"{stock} - Precio histórico de la acción", fontsize=12)
    ax1.set_ylabel("Precio de cierre (USD)")
    ax2.plot(ingresos["Date"], ingresos["Revenue"], color="#16a34a", linewidth=1.8)
    ax2.set_title(f"{stock} - Ingresos históricos trimestrales", fontsize=12)
    ax2.set_ylabel("Ingresos (millones USD)")
    ax2.set_xlabel("Fecha")
    for ax in (ax1, ax2):
        ax.grid(alpha=0.2, linestyle="--")
        ax.spines[["top","right"]].set_visible(False)
    fig.autofmt_xdate()
    return fig


def guardar_evidencia_grafico(numero, titulo, codigo, figura, archivo):
    """Añade el código de ejecución sobre el gráfico y guarda evidencia PNG."""
    figura.subplots_adjust(top=0.84, bottom=0.095, left=0.10, right=0.97)
    figura.suptitle(titulo, fontsize=18, fontweight="bold", y=0.99)
    figura.text(0.10, 0.932, f"Pregunta {numero} · Código ejecutado: {codigo}",
                family="monospace", fontsize=10.5, color="#253f5c")
    ruta = CARPETA / archivo
    figura.savefig(ruta, dpi=165, facecolor="white")
    print("Evidencia guardada:", ruta)
    display(figura)
    plt.close(figura)

## Pregunta 1 — Utilice yfinance para extraer datos bursátiles de Tesla

Obtener el histórico máximo mediante `yf.Ticker`, reiniciar el índice de `tesla_data` y mostrar sus primeras cinco filas con `.head()`. La evidencia debe incluir código y resultados.

In [ ]:
tesla = yf.Ticker("TSLA")
tesla_data = tesla.history(period="max", auto_adjust=False)
if tesla_data.empty:
    raise RuntimeError("Yahoo Finance no devolvió datos de Tesla. Comprueba la conexión.")
tesla_data.reset_index(inplace=True)
display(tesla_data.head())

guardar_evidencia_tabla(
    1, "Datos bursátiles de Tesla",
    'tesla = yf.Ticker("TSLA")\ntesla_data = tesla.history(period="max", auto_adjust=False)\ntesla_data.reset_index(inplace=True)\ntesla_data.head()',
    tesla_data.head(), "01_Tesla_acciones.png"
)

## Pregunta 2 — Utilizar web scraping para extraer ingresos de Tesla

Leer el HTML de IBM con `requests`, localizar con BeautifulSoup la tabla **Tesla Quarterly Revenue**, limpiar los valores y mostrar `tesla_revenue.tail()`.

In [ ]:
url_tesla = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/revenue.htm"
)
html_data = obtener_html(url_tesla)
tesla_revenue = extraer_ingresos_trimestrales(html_data, "Tesla")
display(tesla_revenue.tail())

guardar_evidencia_tabla(
    2, "Ingresos trimestrales de Tesla",
    'html_data = obtener_html(url_tesla)\ntesla_revenue = extraer_ingresos_trimestrales(html_data, "Tesla")\ntesla_revenue.tail()',
    tesla_revenue.tail(), "02_Tesla_ingresos.png"
)

## Pregunta 3 — Utilice yfinance para extraer datos bursátiles de GameStop

Obtener el histórico máximo mediante `yf.Ticker("GME")`, reiniciar el índice de `gme_data` y mostrar `gme_data.head()`.

In [ ]:
gme = yf.Ticker("GME")
gme_data = gme.history(period="max", auto_adjust=False)
if gme_data.empty:
    raise RuntimeError("Yahoo Finance no devolvió datos de GameStop. Comprueba la conexión.")
gme_data.reset_index(inplace=True)
display(gme_data.head())

guardar_evidencia_tabla(
    3, "Datos bursátiles de GameStop",
    'gme = yf.Ticker("GME")\ngme_data = gme.history(period="max", auto_adjust=False)\ngme_data.reset_index(inplace=True)\ngme_data.head()',
    gme_data.head(), "03_GameStop_acciones.png"
)

## Pregunta 4 — Utilizar web scraping para extraer ingresos de GameStop

Leer el HTML histórico de IBM y extraer, limpiar y mostrar las últimas cinco filas de `gme_revenue`.

In [ ]:
url_gme = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/stock.html"
)
html_data = obtener_html(url_gme)
gme_revenue = extraer_ingresos_trimestrales(html_data, "GameStop")
display(gme_revenue.tail())

guardar_evidencia_tabla(
    4, "Ingresos trimestrales de GameStop",
    'html_data = obtener_html(url_gme)\ngme_revenue = extraer_ingresos_trimestrales(html_data, "GameStop")\ngme_revenue.tail()',
    gme_revenue.tail(), "04_GameStop_ingresos.png"
)

## Pregunta 5 — Representar gráficamente las acciones de Tesla

Usar la función `make_graph` con los datos de cotizaciones e ingresos; incluir un título en el gráfico.

In [ ]:
fig_tesla = make_graph(tesla_data, tesla_revenue, "Tesla")
guardar_evidencia_grafico(
    5, "Tesla - Precio de acciones e ingresos",
    'make_graph(tesla_data, tesla_revenue, "Tesla")',
    fig_tesla, "05_Tesla_graficos.png"
)

## Pregunta 6 — Representar gráficamente las acciones de GameStop

Usar la función `make_graph` con las cotizaciones e ingresos de GameStop; incluir un título.

In [ ]:
fig_gme = make_graph(gme_data, gme_revenue, "GameStop")
guardar_evidencia_grafico(
    6, "GameStop - Precio de acciones e ingresos",
    'make_graph(gme_data, gme_revenue, "GameStop")',
    fig_gme, "06_GameStop_graficos.png"
)

## Comprobación y preparación de la entrega

Las siguientes comprobaciones verifican que el Notebook produjo las tablas y las seis imágenes a partir de su ejecución. Se generará el ZIP **`Evidencias_IBM_Tesla_GameStop.zip`** con los seis PNG listos para adjuntar individualmente a las preguntas 1 a 6.

El Notebook `.ipynb` se descarga por separado desde **Archivo → Descargar → Descargar archivo .ipynb** en Colab y se publica en un repositorio público propio de GitHub. Se pega la URL de ese archivo de GitHub en el formulario del curso.

In [ ]:
for nombre, tabla, columnas in [
    ("tesla_data", tesla_data, ["Date", "Close"]),
    ("tesla_revenue", tesla_revenue, ["Date", "Revenue"]),
    ("gme_data", gme_data, ["Date", "Close"]),
    ("gme_revenue", gme_revenue, ["Date", "Revenue"]),
]:
    assert not tabla.empty and set(columnas).issubset(tabla.columns), nombre
    print(f"OK: {nombre}: {len(tabla)} filas")

archivos = sorted(CARPETA.glob("[0-9][0-9]_*.png"))
assert len(archivos) == 6, f"Se esperaban 6 imágenes y solo hay {len(archivos)}"
with ZipFile("Evidencias_IBM_Tesla_GameStop.zip", "w", ZIP_DEFLATED) as paquete:
    for ruta in archivos:
        paquete.write(ruta, arcname=ruta.name)
print("\nLISTO: Evidencias_IBM_Tesla_GameStop.zip")
print("Archivos generados:")
for ruta in archivos:
    print(" •", ruta.name)
print("\nTesla, último ingreso trimestral histórico:", int(tesla_revenue.iloc[-1]["Revenue"]))
print("GameStop, precio de cierre inicial:", float(gme_data.iloc[0]["Close"]))

### Descarga opcional en Google Colab

Ejecuta esta celda en Colab para descargar automáticamente el ZIP con las seis evidencias. Si ejecutas el Notebook en Jupyter local, descarga los archivos desde la carpeta creada.

In [ ]:
try:
    from google.colab import files
    files.download("Evidencias_IBM_Tesla_GameStop.zip")
except ImportError:
    print("No estás en Google Colab. El ZIP está en la carpeta actual.")